# 13. Ландшафт соревнований Kaggle

Считаем динамику основных типов соревнований и распределение по тематическим тегам Meta Kaggle. Темы CV/NLP/tabular/time series определяются словарём проекта и не покрывают все соревнования.

In [ ]:
import sys
sys.path.append("..")
import pandas as pd
import matplotlib.pyplot as plt
from src.data_loader import load
from src.topics import competition_themes

main_types=["Featured","Research","Playground","Getting Started"]
comp=load("Competitions")
comp=comp[comp["HostSegmentTitle"].isin(main_types)].copy()
comp["year"]=comp["EnabledDate"].dt.year
by_year=comp.groupby(["year","HostSegmentTitle"]).size().rename("competitions").reset_index()
by_year.to_csv("../reports/competitions_by_year.csv",index=False)
tags=load("Tags")
ct=load("CompetitionTags").merge(tags,left_on="TagId",right_on="Id")
themes=competition_themes(ct).rename("theme")
profile=comp.set_index("Id").join(themes)
profile["prize_usd"]=profile["RewardQuantity"].where(profile["RewardType"].eq("USD"))
by_theme=profile.groupby("theme",dropna=False).agg(competitions=("Slug","size"),median_teams=("TotalTeams","median"),median_usd_prize=("prize_usd","median")).reset_index()
by_theme["theme"]=by_theme["theme"].fillna("other / unclassified")
by_theme.to_csv("../reports/competition_theme_summary.csv",index=False,float_format="%.2f")
by_year.tail(),by_theme

In [ ]:
pivot=by_year.pivot(index="year",columns="HostSegmentTitle",values="competitions").fillna(0)
ax=pivot.plot(kind="bar",stacked=True,figsize=(11,4),title="Competitions by year and type")
ax.set(xlabel="Year",ylabel="Competitions"); ax.figure.tight_layout()
ax.figure.savefig("../reports/figures/13_competitions_by_year.png",dpi=160)

Год последней выгрузки может быть неполным. Темы присваиваются только по ключевым словам в тегах; нераспознанные соревнования остаются в отдельной группе. Призовой фонд суммируется только для записей с RewardType = USD.